In [5]:
import csv
import pandas as pd
import os

Loading

In [156]:
output_dir = os.path.join("..","final_datasets")
final_hourly_data_path = os.path.join(output_dir, "BTCUSDT_hourly_data_Binance.csv")

In [157]:
with open(final_hourly_data_path,'r') as f:
    reader = csv.reader(f)
    data = list(reader)

In [158]:
df = pd.DataFrame(data, columns=["datetime","open","high","low","close","volume","quote_volume","taker_buy_volume"])

In [159]:
df = df.iloc[1:]

In [160]:
df.dtypes

datetime            object
open                object
high                object
low                 object
close               object
volume              object
quote_volume        object
taker_buy_volume    object
dtype: object

In [161]:
columns = ["open","high","low","close","volume","quote_volume","taker_buy_volume"]
for col in columns:
    df[col] = df[col].astype(float)

df["datetime"] = pd.to_datetime(df["datetime"], format = 'mixed', utc=True) 
df.set_index("datetime", inplace=True)

In [162]:
df

,open,high,low,close,volume,quote_volume,taker_buy_volume
datetime,,,,,,,
2024-06-01 00:00:00+00:00,67540.01,67703.89,67507.39,67655.66,569.87074,3.853123e+07,282.82932
2024-06-01 01:00:00+00:00,67655.66,67710.29,67428.44,67560.00,566.55975,3.827722e+07,227.59647
2024-06-01 02:00:00+00:00,67560.00,67740.00,67546.52,67729.05,327.71958,2.217470e+07,174.19432
2024-06-01 03:00:00+00:00,67729.05,67757.14,67623.24,67715.31,313.20132,2.120333e+07,140.13703
2024-06-01 04:00:00+00:00,67715.31,67763.81,67656.00,67680.01,315.96397,2.139502e+07,136.41640
...,...,...,...,...,...,...,...
2025-04-10 20:00:00+00:00,79647.37,79960.97,79570.12,79931.55,634.20811,5.059605e+07,361.38999
2025-04-10 21:00:00+00:00,79931.56,80035.54,79683.81,79714.52,489.14925,3.905569e+07,259.14683
2025-04-10 22:00:00+00:00,79714.51,79920.00,79545.45,79693.77,413.86755,3.301049e+07,222.57287


Indicator Calculations

In [163]:
df_with_metrics = df.copy() 

In [164]:
import ta 

RSI PER HOUR SUMMARY

In [165]:
df_with_metrics["rsi"] = ta.momentum.RSIIndicator(close=df_with_metrics["close"], window=9).rsi() 

In [166]:
df_with_metrics.iloc[850]

open                5.676599e+04
high                5.687633e+04
low                 5.665000e+04
close               5.678000e+04
volume              7.262433e+02
quote_volume        4.121242e+07
taker_buy_volume    3.614982e+02
rsi                 6.156343e+01
Name: 2024-07-06 10:00:00+00:00, dtype: float64

In [167]:
# Ensure datetime index and add date column
df_with_metrics["date"] = df_with_metrics.index.date

rsi_daily_summary = df_with_metrics.groupby('date').agg(
    rsi_mean=("rsi", "mean"),
    rsi_std=("rsi", "std"),
    rsi_final=("rsi", lambda x: x.dropna().iloc[-1] if not x.dropna().empty else None),
    rsi_overbought_pct=("rsi", lambda x: (x > 70).mean() * 100),
    rsi_oversold_pct=("rsi", lambda x: (x < 30).mean() * 100)
).reset_index() 


In [168]:
rsi_daily_summary

,date,rsi_mean,rsi_std,rsi_final,rsi_overbought_pct,rsi_oversold_pct
0,2024-06-01,55.603512,5.742497,55.843753,0.000000,0.000000
1,2024-06-02,51.895455,9.461543,45.186851,4.166667,0.000000
2,2024-06-03,62.566745,11.031474,44.145910,25.000000,0.000000
3,2024-06-04,59.311449,13.194058,66.986473,25.000000,0.000000
4,2024-06-05,61.474733,8.572607,51.161035,25.000000,0.000000
...,...,...,...,...,...,...
310,2025-04-07,41.799635,15.483517,55.510491,0.000000,25.000000
311,2025-04-08,45.294326,12.811697,25.130784,0.000000,12.500000
312,2025-04-09,53.943718,18.376573,71.019601,29.166667,16.666667
313,2025-04-10,49.970308,12.227646,39.153700,0.000000,4.166667


RSI last 9 days overall (window = 9 days, not 9 periods)

In [169]:
import numpy as np

def compute_rolling_rsi(series, window):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(window=window, min_periods=window).mean()
    avg_loss = loss.rolling(window=window, min_periods=window).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

rsi_daily_summary["rsi_9d"] = compute_rolling_rsi(rsi_daily_summary["rsi_final"], window=9)


In [170]:
rsi_daily_summary 

,date,rsi_mean,rsi_std,rsi_final,rsi_overbought_pct,rsi_oversold_pct,rsi_9d
0,2024-06-01,55.603512,5.742497,55.843753,0.000000,0.000000,NaN
1,2024-06-02,51.895455,9.461543,45.186851,4.166667,0.000000,NaN
2,2024-06-03,62.566745,11.031474,44.145910,25.000000,0.000000,NaN
3,2024-06-04,59.311449,13.194058,66.986473,25.000000,0.000000,NaN
4,2024-06-05,61.474733,8.572607,51.161035,25.000000,0.000000,NaN
...,...,...,...,...,...,...,...
310,2025-04-07,41.799635,15.483517,55.510491,0.000000,25.000000,52.773843
311,2025-04-08,45.294326,12.811697,25.130784,0.000000,12.500000,46.279879
312,2025-04-09,53.943718,18.376573,71.019601,29.166667,16.666667,54.304234
313,2025-04-10,49.970308,12.227646,39.153700,0.000000,4.166667,44.536467


MACD

In [171]:
# Calculate MACD components
macd_ind = ta.trend.MACD(close=df_with_metrics["close"], window_slow=26, window_fast=12, window_sign=9)

df_with_metrics["macd"] = macd_ind.macd()
df_with_metrics["macd_signal"] = macd_ind.macd_signal()
df_with_metrics["macd_diff"] = macd_ind.macd_diff()  # MACD histogram


In [172]:
df_with_metrics

,open,high,low,close,volume,quote_volume,taker_buy_volume,rsi,date,macd,macd_signal,macd_diff
datetime,,,,,,,,,,,,
2024-06-01 00:00:00+00:00,67540.01,67703.89,67507.39,67655.66,569.87074,3.853123e+07,282.82932,NaN,2024-06-01,NaN,NaN,NaN
2024-06-01 01:00:00+00:00,67655.66,67710.29,67428.44,67560.00,566.55975,3.827722e+07,227.59647,NaN,2024-06-01,NaN,NaN,NaN
2024-06-01 02:00:00+00:00,67560.00,67740.00,67546.52,67729.05,327.71958,2.217470e+07,174.19432,NaN,2024-06-01,NaN,NaN,NaN
2024-06-01 03:00:00+00:00,67729.05,67757.14,67623.24,67715.31,313.20132,2.120333e+07,140.13703,NaN,2024-06-01,NaN,NaN,NaN
2024-06-01 04:00:00+00:00,67715.31,67763.81,67656.00,67680.01,315.96397,2.139502e+07,136.41640,NaN,2024-06-01,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-04-10 20:00:00+00:00,79647.37,79960.97,79570.12,79931.55,634.20811,5.059605e+07,361.38999,42.847009,2025-04-10,-190.876178,151.927165,-342.803343
2025-04-10 21:00:00+00:00,79931.56,80035.54,79683.81,79714.52,489.14925,3.905569e+07,259.14683,40.478104,2025-04-10,-221.827167,77.176299,-299.003466
2025-04-10 22:00:00+00:00,79714.51,79920.00,79545.45,79693.77,413.86755,3.301049e+07,222.57287,40.238815,2025-04-10,-245.203844,12.700270,-257.904114


In [173]:
def macd_group_features(group):
    bullish_crosses = ((group["macd"] > group["macd_signal"]).astype(int).diff() == 1).sum()
    bearish_crosses = ((group["macd"] < group["macd_signal"]).astype(int).diff() == 1).sum()

    return pd.Series({
        "macd_final": group["macd"].dropna().iloc[-1] if not group["macd"].dropna().empty else None,
        "macd_signal_final": group["macd_signal"].dropna().iloc[-1] if not group["macd_signal"].dropna().empty else None,
        "macd_diff_mean": group["macd_diff"].mean(),
        "macd_diff_std": group["macd_diff"].std(),
        "macd_hist_final": group["macd_diff"].dropna().iloc[-1] if not group["macd_diff"].dropna().empty else None,
        "macd_bullish_crosses": bullish_crosses,
        "macd_bearish_crosses": bearish_crosses
    })

# Ensure datetime index and add date column
df_with_metrics["date"] = df_with_metrics.index.date

# Apply per group
macd_daily_summary = df_with_metrics.groupby("date").apply(macd_group_features).reset_index()


WEEKLY AND MONTHLY

In [174]:
macd_daily_summary["macd_weekly_hist_final"] = (
    macd_daily_summary["macd_hist_final"]
    .rolling(window=9, min_periods=9)
    .apply(lambda x: x[-1], raw=True)
)

macd_daily_summary["macd_monthly_final"] = (
    macd_daily_summary["macd_final"]
    .rolling(window=30, min_periods=30)
    .apply(lambda x: x[-1], raw=True)
)

macd_daily_summary["macd_monthly_hist_final"] = (
    macd_daily_summary["macd_hist_final"]
    .rolling(window=30, min_periods=30)
    .apply(lambda x: x[-1], raw=True)
)


In [175]:
macd_daily_summary

,date,macd_final,macd_signal_final,macd_diff_mean,macd_diff_std,macd_hist_final,macd_bullish_crosses,macd_bearish_crosses,macd_weekly_hist_final,macd_monthly_final,macd_monthly_hist_final
0,2024-06-01,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN
1,2024-06-02,16.602896,39.936924,3.443711,29.268006,-23.334028,1.0,2.0,NaN,NaN,NaN
2,2024-06-03,195.714274,277.835238,39.649719,59.734399,-82.120964,1.0,1.0,NaN,NaN,NaN
3,2024-06-04,447.378115,376.489434,16.442366,97.104103,70.888681,1.0,0.0,NaN,NaN,NaN
4,2024-06-05,229.365714,289.506890,-14.497091,46.298724,-60.141175,0.0,1.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
310,2025-04-07,-29.340019,-419.537417,64.443062,318.749263,390.197398,1.0,0.0,390.197398,-29.340019,390.197398
311,2025-04-08,-655.287738,-392.235050,4.550395,239.507021,-263.052688,1.0,2.0,-263.052688,-655.287738,-263.052688
312,2025-04-09,1462.388149,851.163238,207.233048,340.838397,611.224911,1.0,0.0,611.224911,1462.388149,611.224911
313,2025-04-10,-267.622450,-43.364274,-149.087919,261.387881,-224.258176,0.0,1.0,-224.258176,-267.622450,-224.258176


BOLLINGER BANDS

In [176]:
from ta.volatility import BollingerBands

bb = BollingerBands(close=df_with_metrics["close"], window=24, window_dev=2)

df_with_metrics["bb_mavg"] = bb.bollinger_mavg()
df_with_metrics["bb_upper"] = bb.bollinger_hband()
df_with_metrics["bb_lower"] = bb.bollinger_lband()
df_with_metrics["bb_width"] = df_with_metrics["bb_upper"] - df_with_metrics["bb_lower"]


In [177]:
def bollinger_features(group):
    return pd.Series({
        "bb_width_mean": group["bb_width"].mean(),                # Average volatility
        "bb_width_std": group["bb_width"].std(),                  # Volatility of volatility
        "bb_upper_touch_pct": (group["close"] >= group["bb_upper"]).mean() * 100,
        "bb_lower_touch_pct": (group["close"] <= group["bb_lower"]).mean() * 100,
    })

bollinger_daily_summary = df_with_metrics.groupby("date").apply(bollinger_features).reset_index()


In [178]:
bollinger_daily_summary 

,date,bb_width_mean,bb_width_std,bb_upper_touch_pct,bb_lower_touch_pct
0,2024-06-01,229.052037,NaN,0.000000,0.000000
1,2024-06-02,505.738165,248.235914,16.666667,8.333333
2,2024-06-03,1864.325165,592.937241,29.166667,0.000000
3,2024-06-04,1612.911832,758.803636,25.000000,0.000000
4,2024-06-05,2107.133619,1036.240967,8.333333,0.000000
...,...,...,...,...,...
310,2025-04-07,7492.345845,1975.009378,0.000000,4.166667
311,2025-04-08,4415.796305,1194.062995,0.000000,16.666667
312,2025-04-09,5791.102614,2299.872260,25.000000,0.000000
313,2025-04-10,7656.894662,2769.790073,0.000000,12.500000


VWAP

In [179]:
df_with_metrics["typical_price"] = (df_with_metrics["high"] + df_with_metrics["low"] + df_with_metrics["close"]) / 3
df_with_metrics["cum_vol_price"] = (df_with_metrics["typical_price"] * df_with_metrics["volume"]).groupby(df_with_metrics["date"]).cumsum()
df_with_metrics["cum_volume"] = df_with_metrics.groupby("date")["volume"].cumsum()
df_with_metrics["vwap"] = df_with_metrics["cum_vol_price"] / df_with_metrics["cum_volume"]

In [180]:
def vwap_features(group):
    price_above_vwap = group["close"] > group["vwap"]

    return pd.Series({
        "vwap_std": group["vwap"].std(),
        "vwap_above_pct": price_above_vwap.mean() * 100,
        "vwap_below_pct": (~price_above_vwap).mean() * 100,
        "vwap_mean_diff": (group["close"] - group["vwap"]).mean(),
        "vwap_abs_diff_mean": (group["close"] - group["vwap"]).abs().mean()
    })

vwap_daily_summary = df_with_metrics.groupby("date").apply(vwap_features).reset_index()


In [181]:
vwap_daily_summary

,date,vwap_std,vwap_above_pct,vwap_below_pct,vwap_mean_diff,vwap_abs_diff_mean
0,2024-06-01,31.381945,87.500000,12.500000,46.522339,53.366353
1,2024-06-02,72.140722,45.833333,54.166667,21.650388,128.766783
2,2024-06-03,371.947445,87.500000,12.500000,226.853496,272.311212
3,2024-06-04,374.795748,70.833333,29.166667,343.729799,400.768188
4,2024-06-05,88.542726,70.833333,29.166667,97.440640,151.688803
...,...,...,...,...,...,...
310,2025-04-07,635.815649,62.500000,37.500000,395.042040,962.783722
311,2025-04-08,476.774470,25.000000,75.000000,-745.960147,907.237582
312,2025-04-09,1186.921600,87.500000,12.500000,1504.808046,1611.056412
313,2025-04-10,597.923506,0.000000,100.000000,-719.578654,719.578654


ADX

In [182]:
from ta.trend import ADXIndicator

# Calculate ADX values (14-period default)
adx_ind = ADXIndicator(high=df_with_metrics["high"], low=df_with_metrics["low"], close=df_with_metrics["close"], window=12)
df_with_metrics["adx"] = adx_ind.adx()


In [183]:
def adx_daily_features(group):
    adx_series = group["adx"].dropna()
    adx_final = adx_series.iloc[-1] if not adx_series.empty else None
    return pd.Series({
        "adx_final": adx_final,
    })

adx_daily_summary = df_with_metrics.groupby("date").apply(adx_daily_features).reset_index()
adx_daily_summary = adx_daily_summary.sort_values("date")
adx_daily_summary["adx_slope"] = adx_daily_summary["adx_final"].diff()


In [184]:
adx_daily_summary 

,date,adx_final,adx_slope
0,2024-06-01,20.386131,NaN
1,2024-06-02,22.479809,2.093679
2,2024-06-03,21.343120,-1.136689
3,2024-06-04,30.570380,9.227260
4,2024-06-05,28.093584,-2.476796
...,...,...,...
310,2025-04-07,36.568261,-14.786965
311,2025-04-08,27.436598,-9.131663
312,2025-04-09,36.822507,9.385909
313,2025-04-10,25.613600,-11.208906


Taker buy ratio 

In [185]:
df_with_metrics["taker_buy_ratio"] = df_with_metrics["taker_buy_volume"] / df_with_metrics["volume"]


In [186]:
taker_buy_daily_summary = (
    df_with_metrics.groupby("date")["taker_buy_ratio"]
    .mean()
    .reset_index(name="taker_buy_ratio_mean")
)


In [187]:
taker_buy_daily_summary 

,date,taker_buy_ratio_mean
0,2024-06-01,0.481797
1,2024-06-02,0.487835
2,2024-06-03,0.503713
3,2024-06-04,0.493266
4,2024-06-05,0.484709
...,...,...
310,2025-04-07,0.499272
311,2025-04-08,0.445990
312,2025-04-09,0.511640
313,2025-04-10,0.503586


COMBINING EVERYTHING

In [199]:
indicators_df = macd_daily_summary.merge(rsi_daily_summary, on="date")
indicators_df = indicators_df.merge(bollinger_daily_summary, on="date")
indicators_df = indicators_df.merge(vwap_daily_summary, on="date") 
indicators_df = indicators_df.merge(adx_daily_summary, on="date") 
indicators_df = indicators_df.merge(taker_buy_daily_summary, on="date")

In [200]:
indicators_df.dtypes

date                        object
macd_final                 float64
macd_signal_final          float64
macd_diff_mean             float64
macd_diff_std              float64
macd_hist_final            float64
macd_bullish_crosses       float64
macd_bearish_crosses       float64
macd_weekly_hist_final     float64
macd_monthly_final         float64
macd_monthly_hist_final    float64
rsi_mean                   float64
rsi_std                    float64
rsi_final                  float64
rsi_overbought_pct         float64
rsi_oversold_pct           float64
rsi_9d                     float64
bb_width_mean              float64
bb_width_std               float64
bb_upper_touch_pct         float64
bb_lower_touch_pct         float64
vwap_std                   float64
vwap_above_pct             float64
vwap_below_pct             float64
vwap_mean_diff             float64
vwap_abs_diff_mean         float64
adx_final                  float64
adx_slope                  float64
taker_buy_ratio_mean

In [202]:
output_dir = os.path.join("..","final_datasets")
output_file = os.path.join(output_dir, "indicators_by_day.csv")

indicators_df.to_csv(output_file, index=False)
print("Data saved to indicators_by_day.csv")


Data saved to indicators_by_day.csv
